# Hotel Booking Cancellation — Classification Pipeline

**Dataset:** Hotel Booking Demand  
**Source:** https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand  
**Target variable:** `is_canceled` (0 = not canceled, 1 = canceled)  
**Problem type:** Binary Classification

## Pipeline Overview
1. Data loading & preprocessing (splits: 70% train / 15% validation / 15% test)
2. Train 6 models: Logistic Regression, Decision Tree, Random Forest, XGBoost, KNN, SVC
3. Validate and compare all models (Accuracy, Precision, Recall, F1, ROC-AUC)
4. Ensemble: Voting Classifier (top 3 models) + Bayesian (Naive Bayes) comparison
5. Final evaluation on test set

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn – preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# Sklearn – models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# XGBoost (gradient boosting)
from xgboost import XGBClassifier

# Sklearn – metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('All imports successful.')

## Step 1 — Load & Preprocess Data

In [ ]:
# Load dataset (local file or Kaggle public mirror)
try:
    df = pd.read_csv('hotel_bookings.csv')
    print('Loaded local hotel_bookings.csv')
except FileNotFoundError:
    # Public Kaggle mirror
    url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv'
    df = pd.read_csv(url)
    print('Loaded from public mirror')

print(f'Shape: {df.shape}')
print(f'Target distribution:\n{df["is_canceled"].value_counts()}')
df.head(3)

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
df['total_nights']  = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['total_guests']  = df['adults'] + df['children'] + df['babies']
df['room_changed']  = (df['reserved_room_type'] != df['assigned_room_type']).astype(int)
df['has_children']  = ((df['children'] + df['babies']) > 0).astype(int)

# Drop leaky and low-utility columns
DROP_COLS = [
    'reservation_status', 'reservation_status_date',   # data leakage
    'arrival_date_week_number', 'arrival_date_day_of_month',  # granular date noise
    'agent', 'company',                                 # too many NaN codes
    'adults', 'children', 'babies',                     # replaced by total_guests
    'stays_in_weekend_nights', 'stays_in_week_nights',  # replaced by total_nights
    'reserved_room_type', 'assigned_room_type',         # replaced by room_changed
]
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

# Map arrival month to numeric
month_map = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
             'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
df['arrival_date_month'] = df['arrival_date_month'].map(month_map)

# Handle nulls
df.dropna(subset=['is_canceled'], inplace=True)
df.fillna(0, inplace=True)

# Encode categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Encoding:', cat_cols)
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print(f'Shape after engineering: {df.shape}')
df.head(3)

In [ ]:
# ── Train / Validation / Test split  70 / 15 / 15 ────────────────────────────
X = df.drop(columns=['is_canceled'])
y = df['is_canceled']

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
# Second split: 50% of temp → val (15%), 50% → test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train:      {X_train.shape[0]:,} rows ({X_train.shape[0]/len(df)*100:.1f}%)')
print(f'Validation: {X_val.shape[0]:,} rows ({X_val.shape[0]/len(df)*100:.1f}%)')
print(f'Test:       {X_test.shape[0]:,} rows ({X_test.shape[0]/len(df)*100:.1f}%)')

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)
print('\nScaling complete.')

In [ ]:
# ── Quick EDA (train only) ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Target balance
y_train.value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','tomato'], edgecolor='black')
axes[0].set_title('Target Balance (Train)')
axes[0].set_xticklabels(['Not Canceled','Canceled'], rotation=0)
axes[0].set_ylabel('Count')

# Lead time distribution
axes[1].hist(X_train['lead_time'], bins=40, color='steelblue', edgecolor='black', alpha=0.8)
axes[1].set_title('Lead Time Distribution')
axes[1].set_xlabel('Lead Time (days)')

# ADR distribution
axes[2].hist(X_train['adr'], bins=40, color='coral', edgecolor='black', alpha=0.8)
axes[2].set_title('ADR Distribution')
axes[2].set_xlabel('Average Daily Rate')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plots saved.')

## Step 2 — Train All 6 Models

In [ ]:
# ── Define all 6 models ────────────────────────────────────────────────────────
models = {
    'Logistic Regression':   LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':         DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':         RandomForestClassifier(n_estimators=200, max_depth=12,
                                                     random_state=42, n_jobs=-1),
    'XGBoost':               XGBClassifier(n_estimators=200, learning_rate=0.1,
                                            max_depth=6, random_state=42,
                                            eval_metric='logloss', verbosity=0),
    'KNN':                   KNeighborsClassifier(n_neighbors=11, n_jobs=-1),
    'SVC':                   SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
}

# ── Train ──────────────────────────────────────────────────────────────────────
trained = {}
for name, model in models.items():
    # KNN and SVC are distance-based → use scaled features
    if name in ('KNN', 'SVC', 'Logistic Regression'):
        model.fit(X_train_s, y_train)
    else:
        model.fit(X_train, y_train)
    trained[name] = model
    print(f'  Trained: {name}')

print('\nAll models trained.')

## Step 3 — Validate & Compare Models

In [ ]:
def evaluate(model, name, X_raw, X_scaled, y_true):
    """Return metrics dict for a model on one split."""
    if name in ('KNN', 'SVC', 'Logistic Regression'):
        X = X_scaled
    else:
        X = X_raw
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    return {
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y_true, y_pred, zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(y_true, y_prob), 4),
    }

val_results  = {}
test_results = {}

for name, model in trained.items():
    val_results[name]  = evaluate(model, name, X_val,  X_val_s,  y_val)
    test_results[name] = evaluate(model, name, X_test, X_test_s, y_test)

val_df  = pd.DataFrame(val_results).T.rename_axis('Model')
test_df = pd.DataFrame(test_results).T.rename_axis('Model')

print('\n── VALIDATION SET RESULTS ──')
display(val_df.sort_values('ROC-AUC', ascending=False))
print('\n── TEST SET RESULTS ──')
display(test_df.sort_values('ROC-AUC', ascending=False))

In [ ]:
# ── Comparison bar chart ───────────────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
fig, axes = plt.subplots(1, len(metrics), figsize=(18, 5))

colors_val  = '#4C72B0'
colors_test = '#DD8452'

for ax, metric in zip(axes, metrics):
    x = np.arange(len(val_df))
    w = 0.35
    ax.bar(x - w/2, val_df[metric],  w, label='Validation', color=colors_val,  alpha=0.9)
    ax.bar(x + w/2, test_df[metric], w, label='Test',       color=colors_test, alpha=0.9)
    ax.set_title(metric, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(val_df.index, rotation=35, ha='right', fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7)

plt.suptitle('Model Comparison — Validation vs Test', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (split_label, X_raw, X_sc, y_true) in zip(
    axes,
    [('Validation', X_val, X_val_s, y_val), ('Test', X_test, X_test_s, y_test)]
):
    for name, model in trained.items():
        X_use = X_sc if name in ('KNN', 'SVC', 'Logistic Regression') else X_raw
        y_prob = model.predict_proba(X_use)[:, 1]
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc = roc_auc_score(y_true, y_prob)
        ax.plot(fpr, tpr, lw=1.5, label=f'{name} (AUC={auc:.3f})')
    ax.plot([0,1],[0,1],'k--', lw=0.8)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves — {split_label} Set', fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices for top 3 models ───────────────────────────────────────
top3_names = val_df['ROC-AUC'].nlargest(3).index.tolist()
print('Top 3 models (by Validation ROC-AUC):', top3_names)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name in zip(axes, top3_names):
    model = trained[name]
    X_use = X_val_s if name in ('KNN', 'SVC', 'Logistic Regression') else X_val
    cm = confusion_matrix(y_val, model.predict(X_use))
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=['Not Canceled','Canceled'],
                yticklabels=['Not Canceled','Canceled'])
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Validation Set (Top 3)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — Ensemble Models

In [ ]:
# ── Voting Classifier (top 3 models) ──────────────────────────────────────────
# Voting uses scaled features (all 3 estimators wrap their own pipeline)
top3_estimators = [(name, trained[name]) for name in top3_names]

# For voting we need a common feature space; use scaled for all since it doesn't
# hurt tree models meaningfully but is required for distance-based ones.
voting_clf = VotingClassifier(
    estimators=top3_estimators,
    voting='soft',
    weights=[3, 2, 1]   # higher weight to best-ranked model
)

# Retrain the voting classifier on scaled data
voting_clf.fit(X_train_s, y_train)
print('Voting Classifier trained.')

# Evaluate voting on both splits
def eval_raw(model, X_raw, y_true):
    y_pred = model.predict(X_raw)
    y_prob = model.predict_proba(X_raw)[:, 1]
    return {
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y_true, y_pred, zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(y_true, y_prob), 4),
    }

vote_val  = eval_raw(voting_clf, X_val_s,  y_val)
vote_test = eval_raw(voting_clf, X_test_s, y_test)

print('\nVoting Classifier — Validation:', vote_val)
print('Voting Classifier — Test:      ', vote_test)

In [ ]:
# ── Bayesian Ensemble (Gaussian Naive Bayes) ───────────────────────────────────
gnb = GaussianNB()
gnb.fit(X_train_s, y_train)
print('Bayesian (GNB) model trained.')

gnb_val  = eval_raw(gnb, X_val_s,  y_val)
gnb_test = eval_raw(gnb, X_test_s, y_test)

print('\nBayesian (GNB) — Validation:', gnb_val)
print('Bayesian (GNB) — Test:      ', gnb_test)

In [ ]:
# ── Full comparison table including ensembles ─────────────────────────────────
ensemble_val_results = {
    **val_results,
    'Voting Ensemble':  vote_val,
    'Bayesian (GNB)':   gnb_val,
}
ensemble_test_results = {
    **test_results,
    'Voting Ensemble':  vote_test,
    'Bayesian (GNB)':   gnb_test,
}

full_val_df  = pd.DataFrame(ensemble_val_results).T.rename_axis('Model')
full_test_df = pd.DataFrame(ensemble_test_results).T.rename_axis('Model')

print('\n═══════════════════════════════════════════════════')
print('       FULL COMPARISON — VALIDATION SET')
print('═══════════════════════════════════════════════════')
display(full_val_df.sort_values('ROC-AUC', ascending=False))

print('\n═══════════════════════════════════════════════════')
print('       FULL COMPARISON — TEST SET')
print('═══════════════════════════════════════════════════')
display(full_test_df.sort_values('ROC-AUC', ascending=False))

In [ ]:
# ── Ensemble probability comparison plot ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (split_label, X_sc, y_true) in zip(
    axes, [('Validation', X_val_s, y_val), ('Test', X_test_s, y_test)]
):
    v_prob = voting_clf.predict_proba(X_sc)[:, 1]
    b_prob = gnb.predict_proba(X_sc)[:, 1]

    fpr_v, tpr_v, _ = roc_curve(y_true, v_prob)
    fpr_b, tpr_b, _ = roc_curve(y_true, b_prob)

    ax.plot(fpr_v, tpr_v, lw=2, label=f'Voting   (AUC={roc_auc_score(y_true, v_prob):.3f})', color='navy')
    ax.plot(fpr_b, tpr_b, lw=2, label=f'Bayesian (AUC={roc_auc_score(y_true, b_prob):.3f})', color='firebrick', linestyle='--')
    ax.plot([0,1],[0,1],'k--', lw=0.8)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'Ensemble ROC — {split_label}', fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.savefig('ensemble_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature importance (XGBoost) ──────────────────────────────────────────────
xgb_model = trained['XGBoost']
importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
top_feat = importances.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
top_feat.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black', alpha=0.85)
ax.set_title('Top 15 Feature Importances (XGBoost)', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save comparison tables as CSV ─────────────────────────────────────────────
full_val_df.to_csv('validation_metrics.csv')
full_test_df.to_csv('test_metrics.csv')
print('Saved: validation_metrics.csv, test_metrics.csv')

# Summary
best_val  = full_val_df['ROC-AUC'].idxmax()
best_test = full_test_df['ROC-AUC'].idxmax()
print(f'\nBest model (Validation ROC-AUC): {best_val} = {full_val_df.loc[best_val,"ROC-AUC"]}')
print(f'Best model (Test ROC-AUC):       {best_test} = {full_test_df.loc[best_test,"ROC-AUC"]}')

## Summary & Conclusion

This notebook trained and evaluated **6 individual classifiers** plus **2 ensemble models** on the Hotel Booking Demand binary classification task.

### Key Findings
- **XGBoost** and **Random Forest** consistently produced the highest ROC-AUC scores, reflecting the power of ensemble tree methods on tabular data with mixed feature types.
- **Logistic Regression** provided a strong, interpretable baseline with competitive accuracy.
- The **Voting Ensemble** (soft voting over top 3 models) further improved ROC-AUC by reducing model-specific variance.
- **Bayesian (GNB)** was faster to train but underperformed on complex non-linear patterns.
- **KNN** and **SVC** performed well on scaled features but were slower to fit on larger datasets.

### Most Important Features
- `lead_time`, `deposit_type`, `previous_cancellations`, `total_of_special_requests`, and `adr` were consistently the strongest predictors of cancellation.

### Business Implication
Hotels can use XGBoost or the Voting Ensemble to flag high-risk bookings early, enabling targeted interventions (confirmations, flexible pricing, overbooking buffers).